# Tutorial 3 — Agrupando municípios com K-Means**Nível:** Iniciante/intermediário · **Tempo:** 30 min**Pergunta:** Municípios podem apresentar perfis de envelhecimento semelhantes mesmo com populações totais diferentes?> Dados didáticos sintéticos, não constituem evidência empírica.

### Como utilizar este notebook

1. Execute as células **de cima para baixo** usando o botão ▶ do Google Colab.
2. Leia a explicação antes de cada código. Não é necessário decorar comandos.
3. Compare os resultados mostrados na saída com as perguntas de interpretação.
4. Experimente as mudanças sugeridas na seção **Agora é sua vez**.

**Convenções de leitura:** `df` representa uma tabela, `X` reúne variáveis preditoras e `y` representa o resultado esperado, quando aplicável.

> **Importante:** todos os dados deste notebook são sintéticos. Os resultados não devem ser empregados para decisões reais.


## Passo 1 — Importar ferramentas

Usaremos um conjunto fictício e duas variáveis numéricas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

## Passo 2 — Gerar municípios fictícios

Os nomes e indicadores são inteiramente sintéticos.

**O que observar:** O gerador cria 45 municípios fictícios. Eles não se referem a nenhum território real.

In [ ]:
rng = np.random.default_rng(12)
n = 45
base = rng.choice([0, 1, 2], size=n)
pct60 = np.clip(8+base*7+rng.normal(0, 2.2, n), 4, 34)
renda = np.clip(1300+base*320+rng.normal(0, 280, n), 550, 3400)
df = pd.DataFrame({"municipio":[f"M{i:02d}" for i in range(1, n+1)], "pct_60mais":pct60, "renda_media":renda})
df.head()

## Passo 3 — Padronizar e agrupar

Escalas distintas afetariam as distâncias sem padronização.

**O que observar:** Sem padronização, a renda, numericamente maior, poderia dominar a distância. `StandardScaler` coloca as variáveis em escalas comparáveis.

In [ ]:
variaveis = ["pct_60mais", "renda_media"]
escalador = StandardScaler()
X = escalador.fit_transform(df[variaveis])
modelo = KMeans(n_clusters=3, random_state=42, n_init=10)
df["grupo"]=modelo.fit_predict(X)
df.groupby("grupo")[variaveis].mean().round(1)

## Passo 4 — Visualizar perfis

Os agrupamentos dependem das variáveis, da escala e do número de grupos.

**O que observar:** Cada ponto representa um município. As cores representam grupos encontrados, não classificações oficiais.

In [ ]:
fig, ax=plt.subplots(figsize=(8, 5))
for k, sub in df.groupby("grupo"):
    ax.scatter(sub.pct_60mais, sub.renda_media, label=f"Grupo {k}", alpha=.8)
ax.set_xlabel("População de 60 anos ou mais (%)")
 ax.set_ylabel("Renda média fictícia")
ax.legend()
 plt.tight_layout()
 plt.show()

## Passo 5 — Comparar números de grupos

Inércia sempre tende a diminuir com mais grupos; não é justificativa isolada para escolher k.

**O que observar:** A inércia informa a compactação interna dos grupos. Ela diminui conforme cresce `k`, por isso requer interpretação combinada com outros critérios.

In [ ]:
for k in [2, 3, 4, 5]:
    m=KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    print(f"k={k} | inércia={m.inertia_:.1f}")

## Experimente você mesmoAltere n_clusters para 2 e 4, examine as médias e proponha nomes substantivos aos perfis sem confundir grupos estatísticos com categorias naturais.## Limitações e cuidados metodológicosResultados variam com normalização, variáveis e k; agrupamento não explica relações causais e não é uma classificação oficial.